# SQL Window Functions — AI Engineering Interview Prep

Covers: RANK, ROW_NUMBER, LAG, LEAD, running totals, moving averages.

In [ ]:
import duckdb
conn = duckdb.connect()

conn.execute("""
CREATE TABLE employees AS SELECT * FROM (VALUES
  (1,'Alice',   'Engineering',95000),(2,'Bob',   'Marketing',75000),
  (3,'Charlie', 'Engineering',85000),(4,'Diana',  'Sales',   70000),
  (5,'Edward',  'Engineering',105000),(6,'Fiona', 'Marketing',82000),
  (7,'George',  'HR',         68000),(8,'Hannah', 'Sales',   91000),
  (9,'Ivan',    'Finance',    88000),(10,'Jane',  'Engineering',115000)
) t(emp_id, name, dept, salary)
""")

conn.execute("""
CREATE TABLE monthly_sales AS SELECT * FROM (VALUES
  (1,'2024-01',12000),(2,'2024-01',8000),(3,'2024-01',15000),
  (1,'2024-02',14000),(2,'2024-02',9500),(3,'2024-02',11000),
  (1,'2024-03',13000),(2,'2024-03',10000),(3,'2024-03',18000),
  (1,'2024-04',16000),(2,'2024-04',8500),(3,'2024-04',20000)
) t(rep_id, month, revenue)
""")

print("Tables created.")

## 1. RANK, DENSE_RANK, ROW_NUMBER

In [ ]:
conn.execute("""
SELECT
    name, dept, salary,
    RANK()        OVER (PARTITION BY dept ORDER BY salary DESC) AS rank,
    DENSE_RANK()  OVER (PARTITION BY dept ORDER BY salary DESC) AS dense_rank,
    ROW_NUMBER()  OVER (PARTITION BY dept ORDER BY salary DESC) AS row_num,
    NTILE(2)      OVER (PARTITION BY dept ORDER BY salary DESC) AS salary_half
FROM employees
ORDER BY dept, rank
""").df()

## 2. Top-N Per Group

In [ ]:
# Top 2 earners per department
conn.execute("""
WITH ranked AS (
    SELECT *,
        DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dr
    FROM employees
)
SELECT name, dept, salary, dr
FROM ranked
WHERE dr <= 2
ORDER BY dept, dr
""").df()

## 3. LAG & LEAD — Period-over-Period

In [ ]:
conn.execute("""
SELECT
    rep_id, month, revenue,
    LAG(revenue, 1)  OVER (PARTITION BY rep_id ORDER BY month) AS prev_month_rev,
    LEAD(revenue, 1) OVER (PARTITION BY rep_id ORDER BY month) AS next_month_rev,
    ROUND(
        (revenue - LAG(revenue) OVER (PARTITION BY rep_id ORDER BY month))
        / LAG(revenue) OVER (PARTITION BY rep_id ORDER BY month) * 100, 1
    ) AS mom_growth_pct
FROM monthly_sales
ORDER BY rep_id, month
""").df()

## 4. Running Totals & Moving Averages

In [ ]:
conn.execute("""
SELECT
    rep_id, month, revenue,
    SUM(revenue)  OVER (PARTITION BY rep_id ORDER BY month
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
    ROUND(AVG(revenue) OVER (PARTITION BY rep_id ORDER BY month
                              ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING), 0) AS moving_avg_3m,
    MAX(revenue)  OVER (PARTITION BY rep_id ORDER BY month) AS running_max
FROM monthly_sales
ORDER BY rep_id, month
""").df()

## 5. FIRST_VALUE, LAST_VALUE, NTH_VALUE

In [ ]:
conn.execute("""
SELECT
    name, dept, salary,
    FIRST_VALUE(salary) OVER (PARTITION BY dept ORDER BY salary DESC
                               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS max_in_dept,
    LAST_VALUE(salary)  OVER (PARTITION BY dept ORDER BY salary DESC
                               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS min_in_dept,
    salary - FIRST_VALUE(salary) OVER (PARTITION BY dept ORDER BY salary DESC
                                        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS gap_to_top
FROM employees
ORDER BY dept, salary DESC
""").df()

## Interview Tips

- **RANK vs DENSE_RANK**: ties in RANK skip numbers (1,1,3); DENSE_RANK doesn't (1,1,2).
- **ROW_NUMBER** assigns unique integers even for ties — order of ties is non-deterministic.
- **OVER()** with empty parens = global window (whole table). Add PARTITION BY and/or ORDER BY to narrow.
- **Frame**: `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = 3-row trailing window.
- **Default frame** when ORDER BY is specified: `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.
- LAST_VALUE needs explicit frame clause or you get the current row, not the actual last.